<a href="https://colab.research.google.com/github/renatofb98/Data_science_projects/blob/main/Pesquisa_PNCP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CÉLULA 1 — instalação (roda uma vez só)
# ============================================================
!pip install pypncp --quiet


# ============================================================
# CÉLULA 2 — imports
# ============================================================
import asyncio
import pandas as pd
from pypncp import PNCPClient
import pypncp.resources.precos as precos_mod


# ============================================================
# CÉLULA 3 — patch de correção do bug da biblioteca
# (corrige campo que às vezes vem como dicionário em vez de string)
# ============================================================
if not getattr(precos_mod.ResultadoItem, "_patched_amparo", False):
    _original_init = precos_mod.ResultadoItem.__init__

    def _init_corrigido(self, **data):
        campo = data.get("amparoLegalCriterioDesempate")
        if isinstance(campo, dict):
            data["amparoLegalCriterioDesempate"] = campo.get("nome", str(campo))
        _original_init(self, **data)

    precos_mod.ResultadoItem.__init__ = _init_corrigido
    precos_mod.ResultadoItem._patched_amparo = True
    print("Patch aplicado.")
else:
    print("Patch já estava aplicado, nada a fazer.")


# ============================================================
# CÉLULA 4 — função de busca, com retry e carga reduzida
# ============================================================
async def buscar_pncp_com_retry(query, uf=None, ano=None, tipos_documento="ata",
                                  max_compras=10, tentativas=3):
    for tentativa in range(1, tentativas + 1):
        try:
            async with PNCPClient(timeout=60) as client:
                resultados = []
                async for p in client.precos.buscar_precos(
                    q=query,
                    tipos_documento=tipos_documento,
                    uf=uf,
                    ano=ano,
                    max_compras=max_compras,
                    paginas_itens=1,
                    delay=2.0,
                ):
                    resultados.append(p)
                df = pd.DataFrame(resultados)
                print(f"'{query}' ({uf or 'BR'}, ano={ano or 'todos'}) -> {len(df)} resultados")
                return df
        except Exception as e:
            print(f"Tentativa {tentativa}/{tentativas} falhou: {repr(e)}")
            if tentativa < tentativas:
                await asyncio.sleep(5)
            else:
                print(f"Desistindo de '{query}' após {tentativas} tentativas.")
                return pd.DataFrame()


# ============================================================
# CÉLULA 5 — função interativa (pergunta os parâmetros)
# ============================================================
def pedir_parametros_busca():
    query = input("Produto/serviço a buscar: ").strip()
    uf = input("UF (ex: GO, ou deixe em branco pra busca nacional): ").strip().upper() or None
    ano_input = input("Ano (ex: 2025, ou deixe em branco pra todos os anos): ").strip()
    ano = int(ano_input) if ano_input else None
    max_compras_input = input("Quantidade máxima de compras a processar (padrão 10): ").strip()
    max_compras = int(max_compras_input) if max_compras_input else 10
    return query, uf, ano, max_compras


# ============================================================
# CÉLULA 6 — uso: roda a busca interativa
# ============================================================
query, uf, ano, max_compras = pedir_parametros_busca()
df_resultado = await buscar_pncp_com_retry(query, uf=uf, ano=ano, max_compras=max_compras)
df_resultado.head()

Patch aplicado.
Produto/serviço a buscar: locacao veiculo
UF (ex: GO, ou deixe em branco pra busca nacional): GO
Ano (ex: 2025, ou deixe em branco pra todos os anos): 
Quantidade máxima de compras a processar (padrão 10): 10
'locacao veiculo' (GO, ano=todos) -> 55 resultados


,descricao,fornecedor,cnpj,valor_unitario,valor_total,quantidade,data,orgao,orgao_nome,link
0,VEICULO TIPO PASSEIO 1.0 (ADM) COM QUILOMETRAG...,BRAGA CHAVES COMERCIO E LOCADORA LTDA,52737173000190,358.7,233155.0,650.0,2025-08-18,10221745000134,MUNICIPIO DE JACAREACANGA,https://pncp.gov.br/app/editais/10221745000134...
1,VEICULO TIPO PASSEIO 1.0 (ASSISNTENCIA) COM QU...,S&S SERVICOS E LOCACOES LTDA,48395326000191,300.0,280800.0,936.0,2025-08-18,10221745000134,MUNICIPIO DE JACAREACANGA,https://pncp.gov.br/app/editais/10221745000134...
2,VEICULO TIPO PASSEIO 1.0 (FME) COM QUILOMETRAG...,LF CONSTRUTORA E LOCADORA LTDA,12796703000157,309.9,290066.4,936.0,2025-08-18,10221745000134,MUNICIPIO DE JACAREACANGA,https://pncp.gov.br/app/editais/10221745000134...
3,VEICULO TIPO PASSEIO 1.0 (FMS) COM QUILOMETRAG...,LF CONSTRUTORA E LOCADORA LTDA,12796703000157,309.9,290066.4,936.0,2025-08-18,10221745000134,MUNICIPIO DE JACAREACANGA,https://pncp.gov.br/app/editais/10221745000134...
4,VEICULO TIPO PASSEIO 1.0 (GBNT) VEICULO TIPO P...,BRAGA CHAVES COMERCIO E LOCADORA LTDA,52737173000190,299.0,194350.0,650.0,2025-08-18,10221745000134,MUNICIPIO DE JACAREACANGA,https://pncp.gov.br/app/editais/10221745000134...


In [3]:
df_resultado.to_excel('pncp_resultados.xlsx', index=False)
print('Resultados exportados para pncp_resultados.xlsx')

Resultados exportados para pncp_resultados.xlsx


In [4]:
from google.colab import files
files.download('pncp_resultados.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>